In [6]:
import os 
import json


list_of_dict = []
for filename in os.listdir("../../Article-Bias-Prediction/data/jsons"): 
    with open(f"../../Article-Bias-Prediction/data/jsons/{filename}", "r") as file:
        list_of_dict += [json.load(file)]

In [7]:
import pandas as pd 

df_raw = pd.DataFrame(list_of_dict)
df_raw.columns

Index(['topic', 'source', 'bias', 'url', 'title', 'date', 'authors', 'content',
       'content_original', 'source_url', 'bias_text', 'ID'],
      dtype='object')

In [8]:
df_raw = df_raw[["bias", "title", "date", "content", "bias_text", "ID", "topic", "source"]]

In [9]:
def get_year(date: str):
    if len(date) == 0:
        return pd.NA
    try: 
        if "-" in date:
            return int(date.split("-")[0])
        elif "/" in date:
            return 2000 + int(date.split("/")[-1])
    except Exception as e:
        print(f"date ({len(date)})")
        print(e)
        return pd.NA
    
df_raw["year"] = df_raw["date"].apply(get_year)

# There are 2 items with date: 0001-11-30
# there is 1 item with date: 2050-09-30

def get_month(date: str):
    if len(date) == 0:
        return pd.NA
    try: 
        if "-" in date:
            return int(date.split("-")[1])
        elif "/" in date:
            return int(date.split("/")[0])
    except Exception as e:
        print(f"{date} ({len(date)})")
        print(e)
        return pd.NA
    
df_raw["month"] = df_raw["date"].apply(get_month)

In [10]:
df_raw["content"].apply(len).describe()

count    37554.000000
mean      5847.792006
std       4001.537702
min        131.000000
25%       3355.000000
50%       4949.000000
75%       7020.000000
max      53992.000000
Name: content, dtype: float64

In [12]:
# Drop NA + drop weird years
import numpy as np 
condition = np.logical_and.reduce((
    df_raw["year"].notna(),
    df_raw["month"].notna(),
    df_raw["year"] != 1,
    df_raw["year"] != 2050,
    df_raw["year"] != 2001, # Not enough elements  
    df_raw["year"] != 2007, # Not enough elements
    df_raw["year"] != 2010,  # Not enough elements
    df_raw["content"].apply(len) <= 7000, # q75
    df_raw["content"].apply(len) >= 3300, # q25
))

df = df_raw[condition]
print(f"size df : {len(df)} ({100 * len(df) / len(df_raw):.0f}% of the raw data)")

size df : 16549 (44% of the raw data)


In [14]:
df["year"].value_counts().sort_index()

year
2012    1321
2013    2381
2014    1812
2015    1490
2016    1819
2017    1805
2018    2024
2019    2106
2020    1791
Name: count, dtype: int64

In [7]:
id_splits = {
    "random": {
        split : pd.read_csv(f"../../Article-Bias-Prediction/data/splits/random/{split}.tsv", sep = "\t")["ID"].to_list()
        for split in ["train", "test", "valid"]
    },
    "media": {
        split : pd.read_csv(f"../../Article-Bias-Prediction/data/splits/media/{split}.tsv", sep = "\t")["ID"].to_list()
        for split in ["train", "test", "valid"]
    }
}

In [8]:
for split in ["train", "test", "valid"]:
    df[f"media_{split}"] = df["ID"].apply(lambda id: id in id_splits["media"][split])
    df[f"random_{split}"] = df["ID"].apply(lambda id: id in id_splits["random"][split])

In [9]:
df.to_csv("./data_agg.csv", index=False)

In [10]:
df.groupby(["year", "bias"]).size().describe()

count      27.000000
mean      612.925926
std       197.722013
min       203.000000
25%       503.500000
50%       554.000000
75%       789.000000
max      1062.000000
dtype: float64

In [11]:
stratification_column =[ "year", "bias"]
samples_per_stratum = 150

df_stratified = (
    df
    .groupby(stratification_column, as_index = True)
    .apply(lambda x : x.sample(n = samples_per_stratum))
    .reset_index()
    .drop(["level_2"], axis = 1)
)

# df_stratified.to_csv("./")

In [12]:
df_stratified.to_csv("./splits/stratified_year_balanced.csv", index = False)

In [22]:
df_inference = (
    df
    .set_index("ID")
    .drop(index = df_stratified.set_index("ID").index)
    .sample(n = 10_000)
    .reset_index()
)

In [25]:
df_inference.to_csv("./splits/dataset_for_inference.csv", index = False)